In [17]:
import pandas as pd
import numpy as np
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
import joblib
import glob
from google.colab import drive

In [18]:
# 1. Mount Google Drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [19]:
print("Loading datasets...")
# 2. Load and Combine all 5 datasets
file_path = "/content/drive/My Drive/Colab Notebooks/IPD/FOOD-DATA-GROUP*.csv"
all_files = glob.glob(file_path)

Loading datasets...


In [20]:
df_list = []
for file in all_files:
    temp_df = pd.read_csv(file)
    df_list.append(temp_df)

In [21]:
# Concatenate into one massive database
full_df = pd.concat(df_list, ignore_index=True)

In [22]:
# Drop redundant index columns if they exist
cols_to_drop = ['Unnamed: 0', 'Unnamed: 0.1']
full_df = full_df.drop(columns=[c for c in cols_to_drop if c in full_df.columns])

In [23]:
# Remove missing values and duplicates
full_df = full_df.dropna(subset=['Caloric Value', 'Protein', 'Carbohydrates', 'Fat'])
full_df = full_df.drop_duplicates(subset=['food'])

In [24]:
print(f"Total Unique Foods Loaded: {len(full_df)}")

Total Unique Foods Loaded: 2395


In [25]:
# --- NEW: AUTOMATED DATA SANITIZATION ---
print("Scrubbing invalid foods from the database...")
ban_words = [
    'infant', 'baby', 'formula', 'toddler',
    'mcdonald', 'burger', 'kfc', 'wendy', 'taco bell', 'pizza',
    'candies', 'beverage', 'alcoholic', 'syrup', 'frosting', 'cake', 'cookie', 'gum'
]
pattern = '|'.join(ban_words)
full_df = full_df[~full_df['food'].str.lower().str.contains(pattern, na=False)]

print(f"Total Foods AFTER cleaning: {len(full_df)}")

Scrubbing invalid foods from the database...
Total Foods AFTER cleaning: 2250


In [26]:
# --- MACHINE LEARNING: K-MEANS CLUSTERING ---
print("\nTraining Unsupervised ML Clustering Model...")

# 3. Select the core macronutrients for the AI to analyze
features = ['Caloric Value', 'Protein', 'Carbohydrates', 'Fat', 'Dietary Fiber']
X = full_df[features]


Training Unsupervised ML Clustering Model...


In [27]:
# 4. Scale the data (Crucial for distance-based algorithms like K-Means)
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [28]:
# 5. Build and Train the K-Means Model
# We will ask the AI to find 4 distinct food groups
num_clusters = 4
kmeans = KMeans(n_clusters=num_clusters, random_state=42, n_init=10)
full_df['ML_Cluster'] = kmeans.fit_predict(X_scaled)

In [29]:
# 6. Analyze what the AI learned (Mapping clusters to human concepts)
print("\n--- AI Cluster Analysis ---")
cluster_mapping = {}

for i in range(num_clusters):
    cluster_data = full_df[full_df['ML_Cluster'] == i]
    avg_protein = cluster_data['Protein'].mean()
    avg_carbs = cluster_data['Carbohydrates'].mean()
    avg_fat = cluster_data['Fat'].mean()
    avg_cals = cluster_data['Caloric Value'].mean()

    # Simple logic to auto-label the AI's clusters
    if avg_protein > avg_carbs and avg_protein > avg_fat:
        label = "High Protein"
    elif avg_carbs > avg_protein and avg_cals < 150:
        label = "Low-Cal Carb/Veggie"
    elif avg_carbs > avg_protein and avg_cals >= 150:
        label = "High Carb"
    else:
        label = "High Fat/Dense"

    cluster_mapping[i] = label
    print(f"Cluster {i} -> Auto-Assigned Label: {label}")
    print(f"   Avg: {avg_cals:.0f} kcal | {avg_protein:.1f}g P | {avg_carbs:.1f}g C | {avg_fat:.1f}g F")
    print(f"   Sample Foods: {', '.join(cluster_data['food'].sample(3).tolist())}\n")


--- AI Cluster Analysis ---
Cluster 0 -> Auto-Assigned Label: Low-Cal Carb/Veggie
   Avg: 117 kcal | 6.5g P | 12.2g C | 4.4g F
   Sample Foods: blueberry pie, cottonseed flour low fat, tortilla chips

Cluster 1 -> Auto-Assigned Label: High Carb
   Avg: 548 kcal | 18.1g P | 94.2g C | 12.2g F
   Sample Foods: yellow corn raw, french beans cooked, breadfruit

Cluster 2 -> Auto-Assigned Label: High Protein
   Avg: 4209 kcal | 335.5g P | 0.0g C | 308.0g F
   Sample Foods: duck meat raw, pork arm picnic cooked, turkey breast raw

Cluster 3 -> Auto-Assigned Label: High Protein
   Avg: 787 kcal | 68.0g P | 3.4g C | 52.2g F
   Sample Foods: boston butt steak raw, pollock raw, beef spleen cooked



In [30]:
# 7. Apply the human-readable labels to the dataframe
full_df['Category'] = full_df['ML_Cluster'].map(cluster_mapping)

In [31]:
# 8. Save the ML artifacts for the Flask Backend
save_path = "/content/drive/MyDrive/Colab Notebooks/IPD/"

# Save the scaler and model
joblib.dump(scaler, save_path + "food_scaler.pkl")
joblib.dump(kmeans, save_path + "food_clustering_model.pkl")

# Save the finalized, categorized database
full_df.to_csv(save_path + "ML_Categorized_Food_Database.csv", index=False)
print("\nSUCCESS! Models and categorized database saved to Drive.")


SUCCESS! Models and categorized database saved to Drive.
